In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

In [2]:
project_root = Path.cwd().parent

raw_tree_path = project_root / "data" / "raw" / "trees"

processed_path = project_root / "data" / "processed"

processed_path.mkdir(exist_ok=True)

In [3]:
def read_tree_tables(data_path):
    """
    Read all raw tree inventory tables.

    Parameters
    ----------
    data_path : Path
        Folder containing tree CSV files.

    Returns
    -------
    dict
        Dictionary of DataFrames.
    """

    datasets = {}

    csv_files = sorted(data_path.glob("*.csv"))

    for file in csv_files:

        datasets[file.stem] = pd.read_csv(file)

    print(f"{len(datasets)} datasets loaded successfully.")

    return datasets

In [4]:
tree_tables = read_tree_tables(raw_tree_path)

9 datasets loaded successfully.


In [5]:
def extract_metadata(filename):
    """
    Extract ecological metadata from filename.
    """

    name = filename.replace("_trees", "")

    parts = name.split("_")

    zone = parts[0].title()

    habitat = " ".join(parts[1:]).title()

    return zone, habitat

In [6]:
for name in tree_tables.keys():

    print(name)

    print(extract_metadata(name))

    print()

buffer_major_river_trees
('Buffer', 'Major River')

buffer_stream_trees
('Buffer', 'Stream')

buffer_upland_trees
('Buffer', 'Upland')

core_major_river_trees
('Core', 'Major River')

core_stream_trees
('Core', 'Stream')

core_upland_trees
('Core', 'Upland')

transition_major_river_trees
('Transition', 'Major River')

transition_stream_trees
('Transition', 'Stream')

transition_upland_trees
('Transition', 'Upland')



In [7]:
assert extract_metadata("core_major_river_trees") == ("Core","Major River")

## Standardize Column Names

The raw datasets were compiled over multiple field campaigns and therefore contain inconsistent column names.

This section standardizes all column names to a common format so that the datasets can be merged without errors.

Examples include:

- SPECIES → Species
- Freq. → Frequency
- total / Individual / Abundance → Individuals
- density / Density → Density
- Rel density / Rel.density → Relative_Density
- Rel. Freq. → Relative_Frequency

In [8]:
def standardize_columns(df):
    """
    Standardize column names across all tree inventory datasets.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw tree inventory table.

    Returns
    -------
    pandas.DataFrame
        DataFrame with standardized column names.
    """

    df = df.copy()

    rename_dict = {

        "SPECIES": "Species",
        "Species": "Species",

        "Freq": "Frequency",
        "Freq.": "Frequency",
        "Frequency": "Frequency",

        "total": "Individuals",
        "Individual": "Individuals",
        "Individuals": "Individuals",
        "Abundance": "Individuals",

        "Density": "Density",
        "density": "Density",

        "Rel density": "Relative_Density",
        "Rel.density": "Relative_Density",
        "Rel. density": "Relative_Density",

        "Rel. frequency": "Relative_Frequency",
        "Rel. Freq.": "Relative_Frequency",
        "Rel. Freq": "Relative_Frequency",

        "RIV": "RIV"

    }

    df.rename(columns=rename_dict, inplace=True)

    return df

## Test Column Standardization

Before applying the function to every dataset, we first test it using one representative dataset.

This confirms that the expected column names are correctly standardized.

In [9]:
sample = standardize_columns(tree_tables["core_major_river_trees"])

sample.columns.tolist()

['Species',
 'CmP1',
 'CmP2',
 'CmP3',
 'CmP4',
 'CmP5',
 'CmP6',
 'CmP7',
 'CmP8',
 'CmP9',
 'CmP10',
 'Frequency',
 'Unnamed: 12',
 'Individuals',
 'Density',
 'Relative_Density',
 'Relative_Frequency',
 'RIV']

## Remove Empty Columns

Some tables contain empty columns introduced during spreadsheet formatting.

These columns contain no information and are removed before merging the datasets.

In [10]:
def remove_empty_columns(df):
    """
    Remove columns that contain only missing values.
    """

    df = df.copy()

    return df.dropna(axis=1, how="all")

In [11]:
sample = remove_empty_columns(sample)

sample.columns.tolist()

['Species',
 'CmP1',
 'CmP2',
 'CmP3',
 'CmP4',
 'CmP5',
 'CmP6',
 'CmP7',
 'CmP8',
 'CmP9',
 'CmP10',
 'Frequency',
 'Individuals',
 'Density',
 'Relative_Density',
 'Relative_Frequency',
 'RIV']

## Remove Text Values from Plot Columns

During data validation, some plot columns were found to contain duplicated species names instead of abundance values.

These entries are data entry artifacts introduced during spreadsheet preparation.

Since plot columns should contain only numeric abundance values, all non-numeric values are replaced with missing values (NaN).

This preserves the original raw data while producing a clean analytical dataset.

In [23]:
def clean_plot_columns(df):
    """
    Convert all plot columns to numeric.

    Any text values found in plot columns are converted to NaN.
    """

    df = df.copy()

    plot_columns = []

    for col in df.columns:

        if "P" in col:
            plot_columns.append(col)

    for col in plot_columns:

        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

## Remove Summary Rows

Each inventory table ends with a summary row (e.g., "26 species") containing totals for the entire table.

These rows do not represent individual species observations and must be removed before merging the datasets.

In [12]:
def remove_summary_rows(df):
    """
    Remove summary rows such as '26 species' or '27 species'.
    """

    df = df.copy()

    df = df[
        ~df["Species"]
        .astype(str)
        .str.contains("species", case=False, na=False)
    ]

    df.reset_index(drop=True, inplace=True)

    return df

## Standardize Plot Column Names

Each dataset contains plot columns with different prefixes (e.g., BmP1, CmP1, TsP1).

These prefixes identify the source dataset but prevent straightforward merging.

Since the management zone and habitat are stored separately as metadata, the plot columns are standardized to a common format (Plot1–Plot10) before combining the datasets.

In [41]:
import re

def standardize_plot_columns(df):
    """
    Rename plot columns to Plot1...Plot10 regardless of dataset prefix.
    """

    df = df.copy()

    rename_dict = {}

    for col in df.columns:

        match = re.match(r"^[A-Za-z]+P\s*([0-9]+(?:\.[0-9]+)?)$", col)

        if match:

            number = match.group(1)

            # Handle TsP5.1 -> Plot6
            if number == "5.1":
                number = "6"

            rename_dict[col] = f"Plot{int(float(number))}"

    df.rename(columns=rename_dict, inplace=True)

    return df

In [42]:
sample = remove_summary_rows(sample)

print(sample.tail())

                  Species  CmP1  CmP2  CmP3  CmP4  CmP5  CmP6  CmP7  CmP8  \
20  Sterculia rhinopetala   NaN   1.0   1.0   3.0   1.0   1.0   1.0   NaN   
21   Strombosia pustulata   3.0   6.0   4.0   2.0   6.0   1.0   NaN   3.0   
22     Trichilia gilgiana   NaN   1.0   NaN   1.0   NaN   NaN   NaN   NaN   
23   Trichilia monadelpha   NaN   NaN   NaN   NaN   NaN   1.0   NaN   NaN   
24      Uapaca heudelotii   NaN   NaN   NaN   NaN   NaN   NaN   NaN   1.0   

    CmP9  CmP10  Frequency  Individuals  Density  Relative_Density  \
20   NaN    NaN          6            8   0.0128            2.9962   
21   NaN    2.0          8           27   0.0432           10.1123   
22   1.0    NaN          3            3   0.0048            1.1235   
23   NaN    2.0          2            3   0.0048            1.1235   
24   NaN    2.0          2            3   0.0048            1.1235   

    Relative_Frequency     RIV  
20              4.9180  3.9571  
21              6.5573  8.3348  
22              2

## Build the Master Database

This section processes every tree inventory table by:

1. Standardizing column names.
2. Removing empty columns.
3. Removing summary rows.
4. Adding ecological metadata (Zone and Habitat).

The cleaned datasets are then merged into a single master database.

In [43]:
master_tables = []

for name, df in tree_tables.items():

    # Metadata
    zone, habitat = extract_metadata(name)

    # Cleaning
    df = standardize_columns(df)
    df = remove_empty_columns(df)
    df = remove_summary_rows(df)
    df = standardize_plot_columns(df)
    df = clean_plot_columns(df)

    # Add metadata
    df["Zone"] = zone
    df["Habitat"] = habitat

    master_tables.append(df)

print(f"{len(master_tables)} cleaned datasets ready for merging.")

9 cleaned datasets ready for merging.


In [44]:
tree_master_database = pd.concat(
    master_tables,
    ignore_index=True
)

print(tree_master_database.shape)

(222, 19)


In [45]:
tree_master_database.head()

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
0,Alstonei boonei,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,2,2,0.0027,1.1560,2.0400,1.5984,Buffer,Major River
1,Blighia sapida,2.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN,1.0,1.0,5,5,0.0068,2.8901,5.1020,3.9961,Buffer,Major River
2,Bombax buonopozene,NaN,NaN,NaN,2.0,1.0,2.0,1.0,NaN,1.0,NaN,5,7,0.0096,4.0462,5.1020,4.5741,Buffer,Major River
3,Bosqueia angolensis,NaN,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,NaN,3,4,0.0055,2.3121,3.0612,2.6866,Buffer,Major River
4,Celtis zenkeri,NaN,1.0,NaN,1.0,1.0,NaN,1.0,2.0,NaN,NaN,4,5,0.0068,2.8901,4.0816,3.4859,Buffer,Major River


## Validate the Master Database

The merged dataset is checked to confirm:

- Number of records
- Number of variables
- Distribution of observations by Zone
- Distribution of observations by Habitat

In [46]:
print("Dataset Shape:", tree_master_database.shape)

print("\nSpecies Records by Zone")
print(tree_master_database["Zone"].value_counts())

print("\nSpecies Records by Habitat")
print(tree_master_database["Habitat"].value_counts())

Dataset Shape: (222, 19)

Species Records by Zone
Zone
Buffer        87
Core          73
Transition    62
Name: count, dtype: int64

Species Records by Habitat
Habitat
Major River    78
Stream         73
Upland         71
Name: count, dtype: int64


## Data Quality Inspection

Before exporting the master database, we inspect the merged data for unexpected values and inconsistencies that could affect subsequent analyses.

In [47]:
# Display column data types

tree_master_database.dtypes

Species                object
Plot1                 float64
Plot2                 float64
Plot3                 float64
Plot4                 float64
Plot5                 float64
Plot6                 float64
Plot7                 float64
Plot8                 float64
Plot9                 float64
Plot10                float64
Frequency               int64
Individuals             int64
Density               float64
Relative_Density      float64
Relative_Frequency    float64
RIV                   float64
Zone                   object
Habitat                object
dtype: object

In [49]:
# Find non-numeric values in BmP1

# tree_master_database[
#     pd.to_numeric(
#         tree_master_database["BmP1"],
#         errors="coerce"
#     ).isna()
# ][["Species","BmP1"]]

## Validate Plot Columns

The plot columns should contain only numeric values representing the number of individuals observed in each sampling plot.

This section identifies any unexpected text values that may have resulted from spreadsheet entry or formatting errors.

In [30]:
# Identify all plot columns
plot_columns = []

for col in tree_master_database.columns:
    if "P" in col and col not in ["Species"]:
        plot_columns.append(col)

print(f"{len(plot_columns)} plot columns found.")
print(plot_columns)

90 plot columns found.
['BmP1', 'BmP2', 'BmP3', 'BmP4', 'BmP5', 'BmP6', 'BmP7', 'BmP8', 'BmP9', 'BmP10', 'BsP1', 'BsP2', 'BsP3', 'BsP4', 'BsP5', 'BsP6', 'BsP7', 'BsP8', 'BsP9', 'BsP10', 'BuP1', 'BuP2', 'BuP3', 'BuP4', 'BuP5', 'BuP6', 'BuP7', 'BuP8', 'BuP9', 'BuP10', 'CmP1', 'CmP2', 'CmP3', 'CmP4', 'CmP5', 'CmP6', 'CmP7', 'CmP8', 'CmP9', 'CmP10', 'CSP1', 'CsP2', 'CsP3', 'CsP4', 'CsP5', 'CsP6', 'CsP7', 'CsP8', 'CsP9', 'CsP10', 'CuP1', 'CuP2', 'CuP3', 'CuP4', 'CuP5', 'CuP6', 'CuP7', 'CuP8', 'CuP9', 'CuP10', 'TmP1', 'TmP2', 'TmP3', 'TmP4', 'TmP5', 'TmP6', 'TmP7', 'TmP8', 'TmP9', 'TmP10', 'TsP1', 'TsP2', 'TsP3', 'TsP4', 'TsP5', 'TsP5.1', 'TsP7', 'TsP8', 'TsP9', 'TsP 10', 'TuP1', 'TuP2', 'TuP3', 'TuP4', 'TuP5', 'TuP6', 'TuP7', 'TuP8', 'TuP9', 'TuP10']


In [31]:
# Find non-numeric values in all plot columns

problems = {}

for col in plot_columns:

    bad = tree_master_database[
        tree_master_database[col].notna() &
        pd.to_numeric(tree_master_database[col], errors="coerce").isna()
    ][["Species", col]]

    if len(bad) > 0:
        problems[col] = bad

print(f"{len(problems)} problematic columns detected.")

0 problematic columns detected.


In [33]:
for column, bad_rows in problems.items():

    print("="*70)
    print(column)
    print("="*70)

    display(bad_rows)

## Validate Plot Abundance Values

This section validates that all plot columns contain valid abundance counts.

The checks include:

- Negative values
- Decimal values (plot counts should be whole numbers)
- Missing values

In [35]:
# Extract all plot columns

plot_columns = [col for col in tree_master_database.columns if "P" in col]

plot_data = tree_master_database[plot_columns]

plot_data.head()

,BmP1,BmP2,BmP3,BmP4,BmP5,BmP6,BmP7,BmP8,BmP9,BmP10,...,TuP1,TuP2,TuP3,TuP4,TuP5,TuP6,TuP7,TuP8,TuP9,TuP10
0,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,2.0,1.0,2.0,1.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,1.0,NaN,1.0,1.0,NaN,1.0,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
# Check for negative values

negative_values = (plot_data < 0).sum().sum()

print(f"Negative values found: {negative_values}")

Negative values found: 0


In [38]:
# Check for decimal values (ignoring NaNs)

plot_numeric = plot_data.dropna(how="all")

decimal_values = (
    plot_numeric.dropna()
    .apply(lambda col: (col % 1 != 0).sum())
    .sum()
)

print(f"Decimal values found: {decimal_values}")

Decimal values found: 0


In [39]:
# Missing values (species absent from plots)

missing_values = plot_data.isna().sum().sum()

print(f"Missing values (species absence): {missing_values}")

Missing values (species absence): 19167


In [50]:
tree_master_database.columns.tolist()

['Species',
 'Plot1',
 'Plot2',
 'Plot3',
 'Plot4',
 'Plot5',
 'Plot6',
 'Plot7',
 'Plot8',
 'Plot9',
 'Plot10',
 'Frequency',
 'Individuals',
 'Density',
 'Relative_Density',
 'Relative_Frequency',
 'RIV',
 'Zone',
 'Habitat']

## Validate Standardized Plot Columns

After standardizing the plot names, each plot column should contain only numeric abundance values.

The following validation identifies any unexpected text values.

In [51]:
plot_columns = [f"Plot{i}" for i in range(1, 11)]

plot_columns

['Plot1',
 'Plot2',
 'Plot3',
 'Plot4',
 'Plot5',
 'Plot6',
 'Plot7',
 'Plot8',
 'Plot9',
 'Plot10']

In [52]:
problems = {}

for col in plot_columns:

    bad = tree_master_database[
        tree_master_database[col].notna() &
        pd.to_numeric(tree_master_database[col], errors="coerce").isna()
    ][["Species", col]]

    if len(bad):

        problems[col] = bad

print(f"{len(problems)} problematic plot columns found.")

0 problematic plot columns found.


In [53]:
for col, bad in problems.items():

    print("="*60)
    print(col)
    print("="*60)

    display(bad)

In [54]:
tree_master_database.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222 entries, 0 to 221
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Species             222 non-null    object 
 1   Plot1               70 non-null     float64
 2   Plot2               77 non-null     float64
 3   Plot3               74 non-null     float64
 4   Plot4               79 non-null     float64
 5   Plot5               59 non-null     float64
 6   Plot6               102 non-null    float64
 7   Plot7               87 non-null     float64
 8   Plot8               80 non-null     float64
 9   Plot9               89 non-null     float64
 10  Plot10              96 non-null     float64
 11  Frequency           222 non-null    int64  
 12  Individuals         222 non-null    int64  
 13  Density             222 non-null    float64
 14  Relative_Density    222 non-null    float64
 15  Relative_Frequency  222 non-null    float64
 16  RIV     

## Export Master Database

The validated master database is exported as a CSV file.

This dataset serves as the canonical input for all subsequent analyses in the project.

In [55]:
# Export master database

output_file = processed_path / "tree_master_database_v1.0.csv"

tree_master_database.to_csv(
    output_file,
    index=False
)

print(f"Master database successfully saved to:\n{output_file}")

Master database successfully saved to:
C:\Peter\SDM_data\omo_ecodb\omo-ecological-informatics\data\processed\tree_master_database_v1.0.csv


## Verify Saved Dataset

To ensure the export was successful, the saved CSV is reloaded and its dimensions are compared with the in-memory DataFrame.

In [56]:
# Read the saved file back

saved_master = pd.read_csv(output_file)

print("Original Shape :", tree_master_database.shape)
print("Saved Shape    :", saved_master.shape)

Original Shape : (222, 19)
Saved Shape    : (222, 19)


In [57]:
saved_master.head()

,Species,Plot1,Plot2,Plot3,Plot4,Plot5,Plot6,Plot7,Plot8,Plot9,Plot10,Frequency,Individuals,Density,Relative_Density,Relative_Frequency,RIV,Zone,Habitat
0,Alstonei boonei,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,2,2,0.0027,1.1560,2.0400,1.5984,Buffer,Major River
1,Blighia sapida,2.0,NaN,1.0,1.0,NaN,NaN,1.0,NaN,1.0,1.0,5,5,0.0068,2.8901,5.1020,3.9961,Buffer,Major River
2,Bombax buonopozene,NaN,NaN,NaN,2.0,1.0,2.0,1.0,NaN,1.0,NaN,5,7,0.0096,4.0462,5.1020,4.5741,Buffer,Major River
3,Bosqueia angolensis,NaN,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,NaN,3,4,0.0055,2.3121,3.0612,2.6866,Buffer,Major River
4,Celtis zenkeri,NaN,1.0,NaN,1.0,1.0,NaN,1.0,2.0,NaN,NaN,4,5,0.0068,2.8901,4.0816,3.4859,Buffer,Major River


In [58]:
print(f"Unique Species : {saved_master['Species'].nunique()}")
print(f"Zones          : {saved_master['Zone'].unique()}")
print(f"Habitats       : {saved_master['Habitat'].unique()}")

Unique Species : 141
Zones          : ['Buffer' 'Core' 'Transition']
Habitats       : ['Major River' 'Stream' 'Upland']
